# Section 2.5 — The Harmonic Oscillator (Weinberg §2.5)

**What this section claims**
- 3D isotropic H = p²/2m + mω²r²/2 separates into three 1D oscillators: E = ℏω(n_x+n_y+n_z+3/2), degeneracy (N+1)(N+2)/2, N = n_x+n_y+n_z.
- Ladder operators a = √(mω/2ℏ)(x + ip/mω): [a, a†] = 1 and H = ℏω(a†a + 1/2) ⇒ E_n = ℏω(n + 1/2).
-ψₙ(ξ) = (mω/πℏ)^¼ (1/√(2ⁿn!)) Hₙ(ξ)e^{−ξ²/2} with ξ = x√(mω/ℏ) — Hermite polynomials.
- x ∝ (a + a†) connects only n ↔ n±1: radiative selection rule Δn = ±1 (and its matrix elements, needed for transition rates).
- In spherical coordinates the same levels are labeled N = 2n_r + l — the Cartesian/spherical degeneracy bookkeeping agrees.


## Ladder algebra: [a, a†] = 1, H = ℏω(a†a + 1/2) (symbolic + matrix check)

From a, a† the ground state obeys aψ₀ = 0, a first-order ODE whose solution is the Gaussian:


In [ ]:
import sympy as sp

m, w, hbar = sp.symbols('m omega hbar', positive=True)
x = sp.symbols('x', real=True)
psi = sp.Function('psi')
# a psi_0 = 0  <=>  (x + i p/mw) psi = 0  <=>  psi' + (m w / hbar) x psi = 0
sol = sp.dsolve(sp.Derivative(psi(x), x) + m*w/hbar*x*psi(x), psi(x))
print("ground state from a|0> = 0:", sol.rhs, "  (Gaussian, as claimed)")

# verify commutation and spectrum in a truncated Fock basis
import numpy as np
dim = 60
a = np.zeros((dim, dim)); a[np.arange(dim-1), np.arange(1, dim)] = np.sqrt(np.arange(1, dim))
adag = a.T
comm = a@adag - adag@a   # exactly  I - dim*|dim-1><dim-1|  in the truncation
print("[a, a^dag] = 1 up to the truncation corner:",
      np.allclose(comm, np.eye(dim) - np.diag([0]*(dim-1) + [dim])))
Hmat = adag@a + 0.5*np.eye(dim)
ev = np.linalg.eigvalsh(Hmat)
print("lowest 6 eigenvalues (units hbar*w):", np.round(ev[:6], 10))
print("spacing uniform = hbar*w:", np.allclose(np.diff(ev[:20]), 1.0))

## Hermite-polynomial wave functions solve the 1D Schrödinger equation (symbolic)


In [ ]:
n = sp.symbols('n', integer=True, nonnegative=True)
x = sp.symbols('x', real=True)
Hn = sp.hermite(n, x)                       # physicists' Hermite: H_n(x)
def psi_n(nv):
    return (m*w/(sp.pi*hbar))**sp.Rational(1,4) / sp.sqrt(2**nv*sp.factorial(nv)) \
           * sp.exp(-m*w*x**2/(2*hbar)) * sp.hermite(nv, sp.sqrt(m*w/hbar)*x)
def sch_residual(nv):
    ps = psi_n(nv)
    return sp.simplify(-hbar**2/(2*m)*sp.diff(ps, x, 2) + sp.Rational(1,2)*m*w**2*x**2*ps
                       - hbar*w*(nv+sp.Rational(1,2))*ps)
for nv in range(5):
    print(f"n={nv}: residual of (H - E_n) psi_n = {sch_residual(nv)}")

## Selection rules: <n′|x|n> ∝ δ_{n′,n±1} (symbolic + matrix)


In [ ]:
# exact closed form for the ground-state matrix element
I01 = sp.integrate(psi_n(1)*x*psi_n(0), (x, -sp.oo, sp.oo))
print("<1|x|0> =", sp.simplify(I01), " = sqrt(hbar/(2 m w))  [as predicted by x = sqrt(hbar/2mw)(a + a^dag)]")

# full pattern in the truncated Fock basis
X = (adag + a)/np.sqrt(2)     # units sqrt(hbar/(m w)) = 1
X[np.abs(X) < 1e-12] = 0
print("nonzero pattern of <n'|x|n> (rows n', cols n, first 6x6):")
print((X[:6, :6] != 0).astype(int))
print("magnitudes row n+1, col n:", np.round(X[np.arange(1,6), np.arange(0,5)], 8), " = sqrt(n+1)")

## Numerics: independent finite-difference spectrum check + 3D degeneracy bookkeeping


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import eigsh
from scipy.special import eval_hermite, factorial as ff

# -1/2 psi'' + 1/2 x^2 psi = E psi  (hbar = m = w = 1):  diag 1/dx^2 + x^2/2, off -1/(2 dx^2)
Npts, L = 500, 18.0
xs = np.linspace(-L, L, Npts); dx = xs[1]-xs[0]
main = np.ones(Npts)/dx**2 + 0.5*xs**2
Hfd = diags([-0.5*np.ones(Npts-1)/dx**2, main, -0.5*np.ones(Npts-1)/dx**2], [-1, 0, 1])
vals = eigsh(Hfd, k=8, which='SA', return_eigenvectors=False)
print("FD eigenvalues:", np.round(np.sort(vals), 5))
print("exact (n+1/2): ", np.round(np.arange(8)+0.5, 5))

fig, ax = plt.subplots(figsize=(7, 4))
xx = np.linspace(-6.5, 6.5, 900)
ax.plot(xx, 0.5*xx**2, color='0.85', lw=1, label='V(x)')
for nv in range(5):
    ps = np.pi**-0.25/np.sqrt(2**nv*ff(nv))*np.exp(-xx**2/2)*eval_hermite(nv, xx)
    ax.plot(xx, ps + (nv+0.5), lw=1.5, label=f'n={nv}')
    ax.axhline(nv+0.5, color='0.9', lw=0.5)
ax.set_xlabel('x (units sqrt(hbar/mw))'); ax.set_ylabel('psi_n + E_n (units hbar*w)')
ax.set_ylim(-0.2, 5.8); ax.legend(ncol=3, fontsize=8)
plt.tight_layout(); plt.show()

print()
print("3D degeneracy:  N=0: (N+1)(N+2)/2 = 1  (l=0)")
print("                N=1: 3  (l=1)   |   N=2: 6  (l=0,2)")

## My summary
*(fill in — own words, ≤ 5 sentences)*

## Gaps
*(questions to raise in review)*
